In [2]:
"""GPR Campaign Ploemeur — 06-06-2016.  Equivalent to seq06.m."""
from pathlib import Path
import sys
import importlib
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from gdp.data_io import load_mala
from gdp.preprocessing.filtering import filter_data, remove_mean
from gdp.preprocessing.gain import linear_gain
from gdp.preprocessing.image_processing import remove_svd
from gdp.preprocessing.trace_ops import align_traces

sys.path.insert(0, str(Path.cwd()))
import helper_functions.KirchhoffPylopsZeroOffset as KirchhoffPylopsZeroOffset
importlib.reload(KirchhoffPylopsZeroOffset)
from pylops.utils.wavelets import ricker
from helper_functions.migration import (
    gazdag_migration, write_backprop_files,
    lowpass_filter_excitation, dispersion_limited_cutoff,
)

# Loading in data from 6 June 2016

In [3]:
DATA = Path.cwd() / 'fielddata' / 'raw_data' / '060616'
OUT_DIR = Path.cwd() / 'fielddata' / 'output'

# Process Reference

In [4]:
runs = list(range(0, 6)) + list(range(7, 39))

# MATLAB: R = D(:,:,1) — first profile is reference
ref_run = runs[0]
data_runs = runs[1:]

_prof_name = lambda n: 'prof' if n == 0 else f'prof{n}'
ref_raw, info = load_mala(str(DATA / _prof_name(ref_run)), return_object=False)
sf = info['frequency (GHz)']
n_traces = ref_raw.shape[1]
samples = ref_raw.shape[0]

# Preprocess reference
ref_bp = filter_data(ref_raw, fq=(0.02, 0.2), sfreq=sf, btype='bandpass')
ref_dc, _ = remove_mean(ref_bp, 299, 517)

ref_aligned, _, _ = align_traces(ref_dc, ref_dc, upsample=5, normalize=False, align_reference=True)

ref_svd, _ = remove_svd(ref_aligned, low_s=0, high_s=1)

t = np.arange(1, samples + 1) / sf
v = 0.10
dL = 0.05
max_d = 85.0
sc = 1.2
rad_cut = 300
depth = np.linspace(max_d, max_d - n_traces * dL, n_traces)
radius = np.linspace(0, v * 450 / 2, samples)

OUT_DIR.mkdir(exist_ok=True)
(OUT_DIR / 'processed').mkdir(exist_ok=True)

# Migration setup
t_mig  = t[:rad_cut]
f0_mig = 0.1                                        # centre frequency [GHz] — adjust to antenna

# Ricker wavelet (matches PylopsKirchoffMigration convention)
_period = 1.0 / f0_mig
_n_wav  = int(np.ceil(6 * _period / (t_mig[1] - t_mig[0])))
if _n_wav % 2 == 0:
    _n_wav += 1
wav_mig, _, wcenter_mig = ricker(t_mig[:_n_wav], f0=f0_mig)

# Image axes: x = radial distance from borehole [m], z spans borehole depth range
x_img = np.linspace(0, v * t_mig[-1] / 2, rad_cut)

(OUT_DIR / 'migrated').mkdir(exist_ok=True)

# Back-propagation gprMax grid -- lambda/20 at f0_mig (~20 cells/wavelength)
bp_dx           = v / (f0_mig * 20)                      # grid spacing [m]
bp_pml          = 10                                      # PML cells
bp_src_y        = (bp_pml + 1) * bp_dx                   # source just inside PML on borehole side [m]
bp_domain_y     = float(x_img[-1]) + 2 * bp_pml * bp_dx  # radial extent + PML padding both sides [m]
bp_edge_exclude = 5                                       # zero outermost N sources each side before filtering

# Process other profiles

In [5]:
for run in data_runs:
    data, _ = load_mala(str(DATA / _prof_name(run)), return_object=False)
    n = min(data.shape[1], n_traces)

    d_bp = filter_data(data[:, :n], fq=(0.02, 0.2), sfreq=sf, btype='bandpass')
    d_dc, _ = remove_mean(d_bp, 299, 517)

    d_aligned, _, _ = align_traces(d_dc, ref_aligned[:, :n], upsample=5, normalize=True, align_reference=False)

    d_svd, _ = remove_svd(d_aligned, low_s=0, high_s=1)
    d_diff = d_svd - ref_svd[:, :n]
    d_gain, _ = linear_gain(d_diff, t)

    # B-scan — Dt[0] = deepest trace (depth[0] = 85 m)
    Dt    = d_gain[:rad_cut, :].T        # (n, rad_cut) = (n_rec, n_t)
    z_bh  = depth[:n]                   # receiver depths [m]; z_bh[0]=85 (deep) → z_bh[-1]≈0
    x_bh  = np.arange(n) * dL          # relative along-borehole positions [m]
    dt_bh = 1.0 / sf                    # time step [ns]

    limit = max(sc * np.max(np.abs(Dt)), 1.0)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(Dt, aspect='auto', cmap='seismic',
              extent=[radius[0], radius[rad_cut - 1], depth[-1], depth[0]],
              vmin=-limit, vmax=limit)
    ax.invert_yaxis()
    ax.set_xlabel('Radial distance from B2 (m)')
    ax.set_ylabel('Depth from top of B1 (m)')
    fig.savefig(OUT_DIR / 'processed' / f'prof_{run}.png', dpi=150)
    plt.close(fig)

    # --- Kirchhoff migration ---
    # z_img decreasing (85→0) so row 0 of m_kir = 85 m
    z_img   = np.linspace(z_bh.max(), z_bh.min(), n)
    recs_bh = np.vstack((np.zeros(n), z_bh))       # (x=0, z=depth) — left-wall geometry

    K = KirchhoffPylopsZeroOffset.Kirchhoff(
        z=z_img, x=x_img, t=t_mig,
        srcs=recs_bh, recs=recs_bh,
        vel=v, wav=wav_mig, wavcenter=wcenter_mig,
        mode='analytic', dynamic=False,
    )
    m_kir = (K.H @ Dt.flatten()).reshape(len(x_img), len(z_img)).T   # (n_depth, n_radial)

    lim_kir = max(sc * np.max(np.abs(m_kir)), 1.0)
    fig_k, ax_k = plt.subplots(figsize=(8, 6))
    ax_k.imshow(m_kir, aspect='equal', cmap='seismic',
                extent=[x_img[0], x_img[-1], z_img[-1], z_img[0]],
                vmin=-lim_kir, vmax=lim_kir)
    ax_k.invert_yaxis()
    ax_k.set_xlabel('Radial distance from borehole (m)')
    ax_k.set_ylabel('Depth (m)')
    fig_k.savefig(OUT_DIR / 'migrated' / f'kirchhoff_{run}.png', dpi=150)
    plt.close(fig_k)
    np.save(OUT_DIR / 'migrated' / f'kirchhoff_{run}.npy', m_kir)

    # --- Gazdag migration ---
    # Rotate: borehole depth → x (along-track), radial distance → z (continuation direction)
    # d_gain[:rad_cut,:] is (n_t, n_x) as required; v_mig = v/2 applied internally
    m_gaz = gazdag_migration(d_gain[:rad_cut, :], x_bh, t_mig, x_img, v)
    # m_gaz: (n_radial, n_depth) → .T gives (n_depth, n_radial); row 0 = z_bh[0] = 85 m
    lim_gaz = max(sc * np.max(np.abs(m_gaz)), 1.0)
    fig_g, ax_g = plt.subplots(figsize=(8, 6))
    ax_g.imshow(m_gaz.T, aspect='equal', cmap='seismic',
                extent=[x_img[0], x_img[-1], z_bh[-1], z_bh[0]],
                vmin=-lim_gaz, vmax=lim_gaz)
    ax_g.invert_yaxis()
    ax_g.set_xlabel('Radial distance from borehole (m)')
    ax_g.set_ylabel('Depth (m)')
    fig_g.savefig(OUT_DIR / 'migrated' / f'gazdag_{run}.png', dpi=150)
    plt.close(fig_g)
    np.save(OUT_DIR / 'migrated' / f'gazdag_{run}.npy', m_gaz.T)

    # --- Back propagation files (peak-normalised and sign-bit) ---
    eps_r    = (0.299792458 / v) ** 2     # permittivity from velocity (c in m/ns)
    t0_ns    = 299.0 / sf                 # direct-wave end, from remove_mean window start
    eps_r_half = 4.0 * eps_r
    f_cut_hz   = dispersion_limited_cutoff(eps_r_half, bp_dx)

    _bp_common = dict(
        tapered_ntr_nt=Dt, dt_ns=dt_bh,
        x_midpoints=z_bh, t0_ns=t0_ns,
        eps_r=eps_r, v_ice=v,
        dx=bp_dx, domain_y=bp_domain_y, src_y=bp_src_y, pml_cells=bp_pml,
    )

    # Peak-normalised version
    write_backprop_files(OUT_DIR, label=f'prof_{run}', slug=f'prof_{run}',
                         sign_bit=False, **_bp_common)
    exc_file = OUT_DIR / 'backprop' / f'prof_{run}' / 'excitation.txt'
    lowpass_filter_excitation(exc_file, f_cut_hz, edge_exclude=bp_edge_exclude)

    # Sign-bit version
    write_backprop_files(OUT_DIR, label=f'prof_{run} (sign-bit)', slug=f'prof_{run}_signbit',
                         sign_bit=True, **_bp_common)
    exc_file_sb = OUT_DIR / 'backprop' / f'prof_{run}_signbit' / 'excitation.txt'
    lowpass_filter_excitation(exc_file_sb, f_cut_hz, edge_exclude=bp_edge_exclude)

  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=1383,  evanescent bins zeroed: 91066/208833 (43.6%)
  nx=461 → nx_pad=13

# Migration

In [6]:
# Frequency spectrum of excitation vs. gprMax dispersion limit
from helper_functions.migration import dispersion_limited_cutoff

exc_path = OUT_DIR / 'backprop' / 'prof_1' / 'excitation.txt'
exc_data = np.loadtxt(exc_path, skiprows=1)
time_s   = exc_data[:, 0]
traces   = exc_data[:, 1:]
dt_s_exc = float(time_s[1] - time_s[0])

# Spectrum of the middle trace (representative)
mid      = traces.shape[1] // 2
spectrum = np.abs(np.fft.rfft(traces[:, mid]))
freq_hz  = np.fft.rfftfreq(len(time_s), dt_s_exc)
freq_ghz = freq_hz * 1e-9

# gprMax dispersion limits for the half-velocity ice material
eps_r_val      = (0.299792458 / v) ** 2
eps_r_half_val = 4.0 * eps_r_val
f_limit_hz  = dispersion_limited_cutoff(eps_r_half_val, bp_dx, safety_factor=1.0)
f_safe_hz   = dispersion_limited_cutoff(eps_r_half_val, bp_dx, safety_factor=0.7)
f_limit_ghz = f_limit_hz * 1e-9
f_safe_ghz  = f_safe_hz  * 1e-9

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(freq_ghz, spectrum / (spectrum.max() + 1e-30), label='middle trace spectrum')
ax.axvline(f_limit_ghz, color='r',      linestyle='--', label=f'hard limit  {f_limit_ghz:.3f} GHz')
ax.axvline(f_safe_ghz,  color='orange', linestyle='--', label=f'safe cutoff {f_safe_ghz:.3f} GHz')
ax.set_xlabel('Frequency (GHz)')
ax.set_ylabel('Normalised amplitude')
ax.set_title(f'Excitation spectrum  (dx={bp_dx:.3f} m, eps_r_half={eps_r_half_val:.1f})')
ax.legend()
ax.set_xlim(0, 5 * f_limit_ghz)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(OUT_DIR / 'excitation_spectrum.png', dpi=150)
plt.show()

print(f'gprMax hard limit : {f_limit_ghz:.4f} GHz')
print(f'Safe cutoff       : {f_safe_ghz:.4f} GHz')
print(f'dx={bp_dx:.4f} m   eps_r_half={eps_r_half_val:.2f}   v_half={v/2:.4f} m/ns')

gprMax hard limit : 0.3333 GHz
Safe cutoff       : 0.2333 GHz
dx=0.0500 m   eps_r_half=35.95   v_half=0.0500 m/ns


C:\Users\Administrator\AppData\Local\Temp\ipykernel_26704\1722971539.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# Save focus-frame snapshots for completed peak-normalised back-propagation runs
import pyvista

# Timing parameters — must match write_backprop_files defaults
N_SNAP   = 30
SNAP_WIN = 1.0   # ns

n_t_bp    = rad_cut
dt_ns_bp  = 1.0 / sf
T_ns_bp   = n_t_bp * dt_ns_bp
t0_ns_bp  = 299.0 / sf
dt_s_bp   = dt_ns_bp * 1e-9

t_focus_ns = T_ns_bp - t0_ns_bp
t_start_ns = max(dt_ns_bp, t_focus_ns - SNAP_WIN)
t_start_s  = t_start_ns * 1e-9
snap_step  = max(1, int((T_ns_bp * 1e-9 - t_start_s) / (max(1, N_SNAP - 1) * dt_s_bp)))

print(f't_focus = {t_focus_ns:.4f} ns  |  t_start = {t_start_ns:.4f} ns  |  snap_step = {snap_step}')

bp_root  = OUT_DIR / 'backprop'
snap_out = OUT_DIR / 'backprop_snapshots'
snap_out.mkdir(exist_ok=True)

# Collect completed peak-normalised runs (skip sign-bit)
completed = {}
for d in sorted(bp_root.iterdir(), key=lambda p: int(p.name.split('_')[1]) if p.name.split('_')[-1].isdigit() else 9999):
    if 'signbit' in d.name or not d.is_dir():
        continue
    snap_dir   = d / f'backprop_{d.name}_snaps'
    snap_files = sorted(snap_dir.glob('bp_snap*.vti'),
                        key=lambda p: int(p.stem.replace('bp_snap', ''))) if snap_dir.exists() else []
    if snap_files:
        completed[d.name] = snap_files
        print(f'  {d.name}: {len(snap_files)} snapshots found')

print(f'\nProcessing {len(completed)} completed run(s)...')

for slug, snap_files in completed.items():
    snap_times_ns = t_start_ns + np.arange(len(snap_files)) * snap_step * dt_ns_bp
    idx_focus     = int(np.argmin(np.abs(snap_times_ns - t_focus_ns))) + 26
    t_actual_ns   = snap_times_ns[idx_focus]

    mesh   = pyvista.read(str(snap_files[idx_focus]))
    nx_c   = mesh.dimensions[0] - 1   # depth cells  (gprMax x)
    ny_c   = mesh.dimensions[1] - 1   # radial cells (gprMax y)
    dx_m   = mesh.spacing[0]
    domain_x = nx_c * dx_m            # borehole depth extent [m]
    domain_y = ny_c * dx_m            # radial extent [m]

    # x-first cell ordering → reshape (ny_c, nx_c), then transpose to (nx_c, ny_c)
    # so rows = depth, cols = radial for imshow
    e_data = np.array(mesh['E-field'])          # (nx_c*ny_c, 3)
    ez     = e_data[:, 2].reshape(ny_c, nx_c).T   # (nx_c, ny_c): rows=depth, cols=radial
    mag    = np.linalg.norm(e_data, axis=1).reshape(ny_c, nx_c).T

    # extent: [radial_min, radial_max, depth_min, depth_max]
    extent = [0, domain_y, 0, domain_x]
    clim_ez = np.percentile(np.abs(ez), 100)

    fig, axes = plt.subplots(1, 2, figsize=(10, 8))

    im0 = axes[0].imshow(mag, aspect='auto', cmap='inferno',
                          extent=extent, origin='lower')
    plt.colorbar(im0, ax=axes[0], label='|E| [V/m]')
    axes[0].invert_yaxis()
    axes[0].set_title(f'{slug}  |  |E|  |  t={t_actual_ns:.3f} ns (snap {idx_focus})')
    axes[0].set_xlabel('Radial distance [m]')
    axes[0].set_ylabel('Depth [m]')
    axes[0].set_ylim(60, 85)
    axes[0].invert_yaxis()

    im1 = axes[1].imshow(ez, aspect='auto', cmap='seismic',
                          extent=extent, origin='lower',
                          vmin=-clim_ez, vmax=clim_ez)
    plt.colorbar(im1, ax=axes[1], label='Ez [V/m]')
    axes[1].invert_yaxis()
    axes[1].set_title(f'{slug}  |  Ez  |  t={t_actual_ns:.3f} ns')
    axes[1].set_xlabel('Radial distance [m]')
    axes[1].set_ylabel('Depth [m]')
    axes[1].set_ylim(60, 85)
    axes[1].invert_yaxis()

    plt.tight_layout()
    out_path = snap_out / f'{slug}_focus.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f'  Saved {out_path.name}')

print('Done.')

t_focus = 0.8713 ns  |  t_start = 0.8713 ns  |  snap_step = 10
  prof_1: 29 snapshots found
  prof_2: 29 snapshots found
  prof_3: 29 snapshots found
  prof_4: 29 snapshots found
  prof_5: 29 snapshots found
  prof_7: 29 snapshots found
  prof_8: 29 snapshots found
  prof_9: 29 snapshots found
  prof_10: 29 snapshots found
  prof_13: 29 snapshots found
  prof_16: 29 snapshots found

Processing 11 completed run(s)...


C:\Users\Administrator\AppData\Local\Temp\ipykernel_26704\809320347.py:87: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  Saved prof_1_focus.png
  Saved prof_2_focus.png
  Saved prof_3_focus.png
  Saved prof_4_focus.png
  Saved prof_5_focus.png
  Saved prof_7_focus.png
  Saved prof_8_focus.png
  Saved prof_9_focus.png
  Saved prof_10_focus.png
  Saved prof_13_focus.png
  Saved prof_16_focus.png
Done.


# Time-lapse Differencing t_n - t_n-1

In [ ]:
# Kirchhoff & Gazdag: image(t_n+1) - image(t_n) for consecutive runs
# Requires the main loop (previous section) to have been re-run so that
# kirchhoff_{run}.npy / gazdag_{run}.npy exist alongside the .png files.

import matplotlib.ticker as ticker

diff_dir = OUT_DIR / 'difference'
diff_dir.mkdir(exist_ok=True)

DIFF_SC        = 1.2    # color-limit scale factor
TICK_SPACING_X = 0.5    # radial-axis tick spacing [m]
TICK_SPACING_Z = 1.0    # depth-axis tick spacing [m]
NOISE_SUPPRESS = 0.15   # suppress values below this fraction of each image's peak

def _consecutive_pairs(method):
    """Yield (run_a, run_b, img_a, img_b) for runs with saved .npy arrays, in data_runs order."""
    avail = [r for r in data_runs if (OUT_DIR / 'migrated' / f'{method}_{r}.npy').exists()]
    for run_a, run_b in zip(avail[:-1], avail[1:]):
        img_a = np.load(OUT_DIR / 'migrated' / f'{method}_{run_a}.npy')
        img_b = np.load(OUT_DIR / 'migrated' / f'{method}_{run_b}.npy')
        n_common = min(img_a.shape[0], img_b.shape[0])
        yield run_a, run_b, img_a[:n_common], img_b[:n_common]

for method, label in [('kirchhoff', 'Kirchhoff'), ('gazdag', 'Gazdag')]:
    pairs = list(_consecutive_pairs(method))
    if not pairs:
        print(f'[{label}] No saved .npy arrays found — re-run the main processing loop first.')
        continue
    print(f'[{label}] {len(pairs)} consecutive pair(s) found')

    for run_a, run_b, img_a, img_b in pairs:
        diff      = img_b - img_a
        z_common  = depth[:diff.shape[0]]
        lim       = max(DIFF_SC * np.max(np.abs(diff)), 1.0)
        diff_disp = np.where(np.abs(diff) < NOISE_SUPPRESS * np.max(np.abs(diff)), 0.0, diff)

        fig, ax = plt.subplots(figsize=(8, 6))
        ax.imshow(diff_disp, aspect='auto', cmap='seismic',
                  extent=[x_img[0], x_img[-1], z_common[-1], z_common[0]],
                  vmin=-lim, vmax=lim)
        ax.invert_yaxis()
        ax.xaxis.set_major_locator(ticker.MultipleLocator(TICK_SPACING_X))
        ax.yaxis.set_major_locator(ticker.MultipleLocator(TICK_SPACING_Z))
        ax.grid(True, color='k', linewidth=0.3, alpha=0.4)
        ax.set_xlabel('Radial distance from borehole (m)')
        ax.set_ylabel('Depth (m)')
        ax.set_title(f'{label} difference: prof_{run_b} − prof_{run_a}')
        out_path = diff_dir / f'{method}_diff_{run_a}_to_{run_b}.png'
        fig.savefig(out_path, dpi=150)
        plt.close(fig)
        print(f'  Saved {out_path.name}')

print('Done.')

In [ ]:
# Back-propagation: Ez_focus(t_n+1) - Ez_focus(t_n) for consecutive completed runs
# Requires the snapshot cell above to have been run first (reuses `completed`,
# t_focus_ns, t_start_ns, snap_step, dt_ns_bp).

FOCUS_IDX_OFFSET  = 26    # must match the offset used in the snapshot cell above
BP_DIFF_PCT       = 97    # percentile used for difference colour limits
BP_NOISE_SUPPRESS = 0.15  # suppress values below this fraction of each image's peak

bp_diff_dir = OUT_DIR / 'backprop_snapshots' / 'difference'
bp_diff_dir.mkdir(parents=True, exist_ok=True)

def _load_focus_ez(snap_files):
    snap_times_ns = t_start_ns + np.arange(len(snap_files)) * snap_step * dt_ns_bp
    idx = min(int(np.argmin(np.abs(snap_times_ns - t_focus_ns))) + FOCUS_IDX_OFFSET,
              len(snap_files) - 1)
    mesh = pyvista.read(str(snap_files[idx]))
    nx_c = mesh.dimensions[0] - 1
    ny_c = mesh.dimensions[1] - 1
    e_data = np.array(mesh['E-field'])
    ez = e_data[:, 2].reshape(ny_c, nx_c).T          # rows=depth, cols=radial
    domain_x = nx_c * mesh.spacing[0]
    domain_y = ny_c * mesh.spacing[0]
    return ez, snap_times_ns[idx], domain_x, domain_y

avail_bp = sorted(completed.keys(), key=lambda s: int(s.split('_')[1]))
if len(avail_bp) < 2:
    print('Fewer than 2 completed back-propagation runs — nothing to difference yet.')
else:
    print(f'{len(avail_bp)} completed run(s): {avail_bp}')
    for slug_a, slug_b in zip(avail_bp[:-1], avail_bp[1:]):
        ez_a, t_a, dom_x, dom_y = _load_focus_ez(completed[slug_a])
        ez_b, t_b, _, _         = _load_focus_ez(completed[slug_b])

        n_x = min(ez_a.shape[0], ez_b.shape[0])
        n_y = min(ez_a.shape[1], ez_b.shape[1])
        diff = ez_b[:n_x, :n_y] - ez_a[:n_x, :n_y]

        extent    = [0, dom_y, 0, dom_x]
        clim      = np.percentile(np.abs(diff), BP_DIFF_PCT)
        diff_disp = np.where(np.abs(diff) < BP_NOISE_SUPPRESS * np.max(np.abs(diff)), 0.0, diff)

        fig, ax = plt.subplots(figsize=(6, 8))
        im = ax.imshow(diff_disp, aspect='auto', cmap='seismic',
                       extent=extent, origin='lower', vmin=-clim, vmax=clim)
        plt.colorbar(im, ax=ax, label='ΔEz [V/m]')
        ax.set_ylim(60, 85)
        ax.invert_yaxis()
        ax.set_xlabel('Radial distance [m]')
        ax.set_ylabel('Depth [m]')
        ax.set_title(f'Back-propagation ΔEz: {slug_b} (t={t_b:.2f} ns) − {slug_a} (t={t_a:.2f} ns)')

        plt.tight_layout()
        out_path = bp_diff_dir / f'{slug_a}_to_{slug_b}_diff.png'
        plt.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f'  Saved {out_path.name}')

print('Done.')

# Time-lapse Differencing t_n - t_reference

In [12]:
# Kirchhoff & Gazdag: image(t_n) - image(t_reference) for all runs vs a fixed baseline.
# Reuses DIFF_SC, TICK_SPACING_X/Z, ticker from the consecutive-pair cell above.

REF_RUN_DIFF   = 1     # run number used as the fixed reference image
NOISE_SUPPRESS = 0.10  # suppress values below this fraction of each image's peak

ref_diff_dir = OUT_DIR / 'difference_from_ref'
ref_diff_dir.mkdir(exist_ok=True)

for method, label in [('kirchhoff', 'Kirchhoff'), ('gazdag', 'Gazdag')]:
    ref_path = OUT_DIR / 'migrated' / f'{method}_{REF_RUN_DIFF}.npy'
    if not ref_path.exists():
        print(f'[{label}] Reference {ref_path.name} not found — skipping.')
        continue
    img_ref = np.load(ref_path)

    avail = [r for r in data_runs
             if r != REF_RUN_DIFF and (OUT_DIR / 'migrated' / f'{method}_{r}.npy').exists()]
    if not avail:
        print(f'[{label}] No non-reference .npy arrays found.')
        continue
    print(f'[{label}] {len(avail)} run(s) vs ref=prof_{REF_RUN_DIFF}')

    for run in avail:
        img_n    = np.load(OUT_DIR / 'migrated' / f'{method}_{run}.npy')
        n_common = min(img_ref.shape[0], img_n.shape[0])
        diff     = img_n[:n_common] - img_ref[:n_common]
        z_common = depth[:n_common]

        lim       = max(DIFF_SC * np.max(np.abs(diff)), 1.0)
        diff_disp = np.where(np.abs(diff) < NOISE_SUPPRESS * np.max(np.abs(diff)), 0.0, diff)

        fig, ax = plt.subplots(figsize=(8, 6))
        ax.imshow(diff_disp, aspect='auto', cmap='seismic',
                  extent=[x_img[0], x_img[-1], z_common[-1], z_common[0]],
                  vmin=-lim, vmax=lim)
        ax.invert_yaxis()
        ax.xaxis.set_major_locator(ticker.MultipleLocator(TICK_SPACING_X))
        ax.yaxis.set_major_locator(ticker.MultipleLocator(TICK_SPACING_Z))
        ax.grid(True, color='k', linewidth=0.3, alpha=0.4)
        ax.set_xlabel('Radial distance from borehole (m)')
        ax.set_ylabel('Depth (m)')
        ax.set_title(f'{label} difference: prof_{run} − prof_{REF_RUN_DIFF}')
        out_path = ref_diff_dir / f'{method}_diff_ref{REF_RUN_DIFF}_to_{run}.png'
        fig.savefig(out_path, dpi=150)
        plt.close(fig)
        print(f'  Saved {out_path.name}')

print('Done.')

[Kirchhoff] 36 run(s) vs ref=prof_1
  Saved kirchhoff_diff_ref1_to_2.png
  Saved kirchhoff_diff_ref1_to_3.png
  Saved kirchhoff_diff_ref1_to_4.png
  Saved kirchhoff_diff_ref1_to_5.png
  Saved kirchhoff_diff_ref1_to_7.png
  Saved kirchhoff_diff_ref1_to_8.png
  Saved kirchhoff_diff_ref1_to_9.png
  Saved kirchhoff_diff_ref1_to_10.png
  Saved kirchhoff_diff_ref1_to_11.png
  Saved kirchhoff_diff_ref1_to_12.png
  Saved kirchhoff_diff_ref1_to_13.png
  Saved kirchhoff_diff_ref1_to_14.png
  Saved kirchhoff_diff_ref1_to_15.png
  Saved kirchhoff_diff_ref1_to_16.png
  Saved kirchhoff_diff_ref1_to_17.png
  Saved kirchhoff_diff_ref1_to_18.png
  Saved kirchhoff_diff_ref1_to_19.png
  Saved kirchhoff_diff_ref1_to_20.png
  Saved kirchhoff_diff_ref1_to_21.png
  Saved kirchhoff_diff_ref1_to_22.png
  Saved kirchhoff_diff_ref1_to_23.png
  Saved kirchhoff_diff_ref1_to_24.png
  Saved kirchhoff_diff_ref1_to_25.png
  Saved kirchhoff_diff_ref1_to_26.png
  Saved kirchhoff_diff_ref1_to_27.png
  Saved kirchhoff_dif

In [ ]:
# Back-propagation: Ez_focus(t_n) - Ez_focus(t_reference) for all completed runs.
# Requires the snapshot cell to have been run (reuses completed, t_focus_ns, etc.).

BP_REF_SLUG       = 'prof_1'   # slug from `completed` dict used as reference
BP_REF_DIFF_PCT   = 97         # percentile for colour limits
BP_NOISE_SUPPRESS = 0.15       # suppress values below this fraction of each image's peak

bp_ref_diff_dir = OUT_DIR / 'backprop_snapshots' / 'difference_from_ref'
bp_ref_diff_dir.mkdir(parents=True, exist_ok=True)

if BP_REF_SLUG not in completed:
    print(f'{BP_REF_SLUG} not in completed dict — run the snapshot cell first.')
else:
    ez_ref, t_ref, dom_x_ref, dom_y_ref = _load_focus_ez(completed[BP_REF_SLUG])
    avail_bp_ref = sorted(
        [s for s in completed if s != BP_REF_SLUG],
        key=lambda s: int(s.split('_')[1])
    )
    print(f'{len(avail_bp_ref)} run(s) vs ref={BP_REF_SLUG}')

    for slug in avail_bp_ref:
        ez_n, t_n, dom_x, dom_y = _load_focus_ez(completed[slug])

        n_x = min(ez_ref.shape[0], ez_n.shape[0])
        n_y = min(ez_ref.shape[1], ez_n.shape[1])
        diff = ez_n[:n_x, :n_y] - ez_ref[:n_x, :n_y]

        extent    = [0, dom_y, 0, dom_x]
        clim      = np.percentile(np.abs(diff), BP_REF_DIFF_PCT)
        diff_disp = np.where(np.abs(diff) < BP_NOISE_SUPPRESS * np.max(np.abs(diff)), 0.0, diff)

        fig, ax = plt.subplots(figsize=(6, 8))
        im = ax.imshow(diff_disp, aspect='auto', cmap='seismic',
                       extent=extent, origin='lower', vmin=-clim, vmax=clim)
        plt.colorbar(im, ax=ax, label='ΔEz [V/m]')
        ax.set_ylim(60, 85)
        ax.invert_yaxis()
        ax.set_xlabel('Radial distance [m]')
        ax.set_ylabel('Depth [m]')
        ax.set_title(f'Back-propagation ΔEz: {slug} (t={t_n:.2f} ns) − {BP_REF_SLUG} (t={t_ref:.2f} ns)')

        plt.tight_layout()
        out_path = bp_ref_diff_dir / f'{BP_REF_SLUG}_to_{slug}_diff.png'
        plt.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f'  Saved {out_path.name}')

print('Done.')

# Fluid Front Movement Estimate

In [ ]:
# ── Helper functions: Riesz transform, monogenic envelope, ROI, WLS phase fit ──────
from scipy.signal.windows import tukey
from matplotlib.patches import Rectangle

def _riesz_2d(img):
    """Full 2D Riesz transform.
    Returns (R_z, R_x): the two real-valued spatial components of the monogenic signal.
      R_z = IFFT2(-i * kz/|k| * FFT2(img))
      R_x = IFFT2(-i * kx/|k| * FFT2(img))
    Uses the same FFT convention as estimate_shift_2d (numpy, no fftshift).
    """
    nz, nx = img.shape
    KZ, KX = np.meshgrid(np.fft.fftfreq(nz), np.fft.fftfreq(nx), indexing='ij')
    K = np.sqrt(KZ**2 + KX**2)
    K[0, 0] = 1.0      # avoid DC divide-by-zero; DC component maps to 0 anyway
    F  = np.fft.fft2(img)
    Rz = np.real(np.fft.ifft2((-1j * KZ / K) * F))
    Rx = np.real(np.fft.ifft2((-1j * KX / K) * F))
    return Rz, Rx

def _monogenic_envelope(img):
    """Amplitude of the monogenic signal: sqrt(f^2 + Rz^2 + Rx^2).
    This is the 2D generalisation of the 1D Hilbert envelope; it is
    phase-invariant and highlights the spatial extent of coherent energy
    regardless of whether the wavelet is at a zero-crossing or a peak.
    """
    Rz, Rx = _riesz_2d(img)
    return np.sqrt(img**2 + Rz**2 + Rx**2)

def _roi_from_envelope(env, threshold_frac=0.5):
    """Tight bounding box of the high-energy region in the envelope image.
    Pixels with env >= threshold_frac * env.max() define the ROI mask;
    returns (z0, z1, x0, x1) as integer row/col indices (z1, x1 exclusive).
    Falls back to the full image if no pixels survive the threshold.
    """
    mask = env >= threshold_frac * env.max()
    rows, cols = np.where(mask)
    if rows.size == 0:
        return 0, env.shape[0], 0, env.shape[1]
    return int(rows.min()), int(rows.max()) + 1, int(cols.min()), int(cols.max()) + 1

def _estimate_shift_2d(base, mon, dz_g, dx_g, kz_cent, force_dz_zero=False, pad_fac=4):
    """WLS 2D phase-plane fit — mirrors estimate_shift_2d from TimeLapse_Processing.ipynb.

    Fits phi(kz, kx) = kz*dz + kx*dx + phi_0 to the cross-spectrum of base and mon,
    weighted by spectral amplitude and restricted to a band around kz_cent.

    Parameters
    ----------
    base, mon   : 2D arrays (n_depth, n_radial), pre-cropped to the ROI
    dz_g        : depth grid spacing [m]   (row spacing, axis 0)
    dx_g        : radial grid spacing [m]  (col spacing, axis 1)
    kz_cent     : dominant wavenumber [rad/m] = 2π * f0 / v
    force_dz_zero : if True, fit only (dx, phi_0) — use for purely lateral motion
    pad_fac     : zero-padding factor before FFT2 for denser kx/kz sampling (default 4)

    Returns (dz_est, dx_est, phi_0) all as floats [m, m, rad].
    """
    Nz, Nx = base.shape
    Nz_pad, Nx_pad = Nz * pad_fac, Nx * pad_fac
    kz_ax = np.fft.fftfreq(Nz_pad, d=dz_g) * 2 * np.pi
    kx_ax = np.fft.fftfreq(Nx_pad, d=dx_g) * 2 * np.pi
    KZ, KX = np.meshgrid(kz_ax, kx_ax, indexing='ij')
    taper = np.outer(tukey(Nz, alpha=0.15), tukey(Nx, alpha=0.15))
    XS  = (np.fft.fft2(base * taper, s=(Nz_pad, Nx_pad)) *
           np.conj(np.fft.fft2(mon  * taper, s=(Nz_pad, Nx_pad))))
    w   = np.abs(XS)
    phi = np.angle(XS)
    band = (np.abs(KZ) < 1.4 * kz_cent) & (np.abs(KX) < 1.4 * kz_cent)
    mask = (w > 0.10 * w.max()) & band & ((np.abs(KZ) + np.abs(KX)) > 0)
    W = w[mask]
    if force_dz_zero:
        A = np.column_stack([KX[mask], np.ones(mask.sum())])
        c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
        return 0.0, float(c[0]), float(c[1])
    A = np.column_stack([KZ[mask], KX[mask], np.ones(mask.sum())])
    c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
    return float(c[0]), float(c[1]), float(c[2])

In [ ]:
# ── Manual pair selection + ROI definition ──────────────────────────────────────────
#
# MANUAL_PAIRS  — list of (run_a, run_b) to analyse.
#
# COMPARE_TO    — None  → use pairs as written (run_a vs run_b)
#                 int   → override run_a for every pair to this run number,
#                         so you compare every profile against one fixed reference.
#                         Useful when consecutive-pair shifts are too small to detect:
#                         e.g. COMPARE_TO=1 gives (1,4), (1,8), (1,10), ...
#
# MANUAL_ROIS   — per-pair ROI in physical metres.
#                 Key is the ORIGINAL (run_a, run_b) regardless of COMPARE_TO.
#                 Format: (z_min_m, z_max_m, x_min_m, x_max_m)
#                 None → auto-detect from monogenic envelope.
#
# SHOW_DIAG     — True: plot cross-spectrum phase + fitted plane for every pair.
#                 Helps diagnose whether there is any detectable kx slope.
#
# ROI_METHOD    — 'kirchhoff' or 'gazdag'.
# FORCE_DZ_ZERO — True: fit only Δx + φ₀ (lateral fluid front).
# ROI_THRESH    — envelope threshold for auto-detected pairs.
# ────────────────────────────────────────────────────────────────────────────────────

MANUAL_PAIRS = [
    (1,  4),
    (1,  8),
    (1,  10),
    (1,  13),
    (1,  15),
    (1,  16),
]

COMPARE_TO = 1      # fix run_a = 1 for all pairs (cumulative from first profile)
                    # set to None to use MANUAL_PAIRS exactly as written

MANUAL_ROIS = {
    (1,  4):  (71, 78, 5.0, 7.0),
    (1,  8):  (71, 78, 5.0, 7.0),
    (1,  10): (71, 78, 5.0, 7.0),
    (1,  13): (71, 78, 5.0, 7.0),
    (1,  15): (71, 78, 5.0, 7.0),
    (1,  16): (71, 78, 5.0, 7.0),
}

ROI_METHOD    = 'kirchhoff'
FORCE_DZ_ZERO = False
ROI_THRESH    = 0.5
SHOW_DIAG     = True   # set False to skip cross-spectrum phase plots

# ────────────────────────────────────────────────────────────────────────────────────

def _phys_to_pix(z_common, x_img_arr, z_min, z_max, x_min, x_max):
    """Convert physical ROI bounds (metres) to pixel indices.
    z_common is DECREASING (row 0 = 85 m deepest).
    Returns (z0, z1, x0, x1), z1/x1 exclusive.
    """
    rows = np.where((z_common >= z_min) & (z_common <= z_max))[0]
    cols = np.where((x_img_arr >= x_min) & (x_img_arr <= x_max))[0]
    if rows.size == 0 or cols.size == 0:
        raise ValueError(
            f'ROI ({z_min}–{z_max} m depth, {x_min}–{x_max} m radial) '
            f'does not intersect the image grid. '
            f'Depth: [{z_common[-1]:.1f}, {z_common[0]:.1f}] m  '
            f'Radial: [{x_img_arr[0]:.2f}, {x_img_arr[-1]:.2f}] m'
        )
    return int(rows[0]), int(rows[-1]) + 1, int(cols[0]), int(cols[-1]) + 1

roi_out = OUT_DIR / 'roi_phase'
roi_out.mkdir(exist_ok=True)

dz_g = dL
dx_g = float(x_img[1] - x_img[0])
kz_c = 2.0 * np.pi * f0_mig / v

def _load_img(method, run):
    p = OUT_DIR / 'migrated' / f'{method}_{run}.npy'
    return np.load(p) if p.exists() else None

results = {}
for run_a, run_b in MANUAL_PAIRS:
    ref_run = COMPARE_TO if COMPARE_TO is not None else run_a
    img_a = _load_img(ROI_METHOD, ref_run)
    img_b = _load_img(ROI_METHOD, run_b)
    if img_a is None or img_b is None:
        print(f'[{ref_run}→{run_b}] Missing .npy — skipping.')
        continue

    n       = min(img_a.shape[0], img_b.shape[0])
    img_a   = img_a[:n];  img_b = img_b[:n]
    z_common = depth[:n]
    diff     = img_b - img_a
    env      = _monogenic_envelope(diff)

    # ROI — look up by original key (run_a, run_b) first, then (ref_run, run_b)
    roi_spec = MANUAL_ROIS.get((run_a, run_b)) or MANUAL_ROIS.get((ref_run, run_b))
    if roi_spec is not None:
        z0, z1, x0, x1 = _phys_to_pix(z_common, x_img, *roi_spec)
        roi_source = 'manual'
    else:
        z0, z1, x0, x1 = _roi_from_envelope(env, ROI_THRESH)
        roi_source = f'auto (thresh={ROI_THRESH})'

    z_roi_top = float(z_common[z0]);     z_roi_bot = float(z_common[z1 - 1])
    x_roi_lo  = float(x_img[x0]);        x_roi_hi  = float(x_img[x1 - 1])

    # ── Difference + envelope figure ────────────────────────────────────────────
    extent = [x_img[0], x_img[-1], z_common[-1], z_common[0]]
    vmax_d = np.percentile(np.abs(diff), 98)
    vmax_e = np.percentile(env, 98)

    fig, (ax_d, ax_e) = plt.subplots(1, 2, figsize=(14, 5))
    im_d = ax_d.imshow(diff, aspect='auto', cmap='RdBu_r',
                        extent=extent, origin='upper',
                        vmin=-vmax_d, vmax=vmax_d)
    plt.colorbar(im_d, ax=ax_d, label='Δ amplitude [a.u.]')
    ax_d.invert_yaxis()
    ax_d.add_patch(Rectangle((x_roi_lo, z_roi_bot),
                              width=x_roi_hi - x_roi_lo, height=z_roi_top - z_roi_bot,
                              lw=1.5, edgecolor='yellow', facecolor='none'))
    ax_d.xaxis.set_major_locator(ticker.MultipleLocator(0.5))
    ax_d.yaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_d.grid(True, color='white', lw=0.3, alpha=0.4)
    ax_d.set_title(f'{ROI_METHOD.capitalize()} diff: prof_{run_b} − prof_{ref_run}  [{roi_source}]')
    ax_d.set_xlabel('Radial distance (m)');  ax_d.set_ylabel('Depth (m)')

    im_e = ax_e.imshow(env, aspect='auto', cmap='inferno',
                        extent=extent, origin='upper', vmin=0, vmax=vmax_e)
    plt.colorbar(im_e, ax=ax_e, label='Monogenic envelope [a.u.]')
    ax_e.invert_yaxis()
    ax_e.add_patch(Rectangle((x_roi_lo, z_roi_bot),
                              width=x_roi_hi - x_roi_lo, height=z_roi_top - z_roi_bot,
                              lw=1.5, edgecolor='cyan', facecolor='none'))
    ax_e.xaxis.set_major_locator(ticker.MultipleLocator(0.5))
    ax_e.yaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_e.grid(True, color='white', lw=0.3, alpha=0.4)
    ax_e.set_title(f'Monogenic envelope  [{roi_source}]')
    ax_e.set_xlabel('Radial distance (m)');  ax_e.set_ylabel('Depth (m)')

    plt.tight_layout()
    fig.savefig(roi_out / f'{ROI_METHOD}_roi_{ref_run}_to_{run_b}.png', dpi=150)
    plt.show();  plt.close(fig)

    # ── WLS phase-plane fit (zero-padded for denser kx/kz sampling) ─────────────
    base_crop = img_a[z0:z1, x0:x1]
    mon_crop  = img_b[z0:z1, x0:x1]
    Nz, Nx    = base_crop.shape
    pad_fac   = 10
    Nz_pad, Nx_pad = Nz * pad_fac, Nx * pad_fac

    kz_ax = np.fft.fftfreq(Nz_pad, d=dz_g) * 2 * np.pi
    kx_ax = np.fft.fftfreq(Nx_pad, d=dx_g) * 2 * np.pi
    KZ, KX = np.meshgrid(kz_ax, kx_ax, indexing='ij')
    taper  = np.outer(tukey(Nz, alpha=0.15), tukey(Nx, alpha=0.15))
    XS     = (np.fft.fft2(base_crop * taper, s=(Nz_pad, Nx_pad)) *
               np.conj(np.fft.fft2(mon_crop * taper, s=(Nz_pad, Nx_pad))))
    w      = np.abs(XS);  phi = np.angle(XS)
    band   = (np.abs(KZ) < 1.5 * kz_c) & (np.abs(KX) < 1.5 * kz_c)
    mask   = (w > 0.10 * w.max()) & band & ((np.abs(KZ) + np.abs(KX)) > 0)

    n_mask = int(mask.sum())
    if n_mask < 3:
        print(f'[{ref_run}→{run_b}] WARNING: only {n_mask} pixels pass the WLS mask '
              f'(crop {Nz}×{Nx} px, padded to {Nz_pad}×{Nx_pad}).  ROI may be too small or SNR too low.')
        dx_est = phi_0 = dz_est = 0.0
    else:
        W = w[mask]
        if FORCE_DZ_ZERO:
            A = np.column_stack([KX[mask], np.ones(n_mask)])
        else:
            A = np.column_stack([KZ[mask], KX[mask], np.ones(n_mask)])
        c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
        if FORCE_DZ_ZERO:
            dz_est, dx_est, phi_0 = 0.0, float(c[0]), float(c[1])
        else:
            dz_est, dx_est, phi_0 = float(c[0]), float(c[1]), float(c[2])

    # ── Diagnostic: cross-spectrum phase + fitted plane ──────────────────────────
    if SHOW_DIAG and n_mask >= 3:
        # Shift to centre-zero frequency for display
        phi_shift = np.fft.fftshift(phi)
        w_shift   = np.fft.fftshift(w)
        kz_disp   = np.fft.fftshift(kz_ax)
        kx_disp   = np.fft.fftshift(kx_ax)

        # Fitted plane at each pixel
        fitted = KX * dx_est + KZ * dz_est + phi_0
        fitted_shift = np.fft.fftshift(fitted)

        # 1-D kx profile: amplitude-weighted mean phase over all kz in band
        phi_1d  = np.zeros(Nx_pad)
        w_1d    = np.zeros(Nx_pad)
        band_kz = np.abs(kz_ax) < 1.5 * kz_c
        for j in range(Nx_pad):
            sel = band_kz & (w[:, j] > 0.10 * w.max())
            if sel.sum() > 0:
                phi_1d[j] = np.average(phi[:, j][sel], weights=w[:, j][sel])
                w_1d[j]   = w[:, j][sel].sum()
        kx_1d_s  = np.fft.fftshift(kx_ax)
        phi_1d_s = np.fft.fftshift(phi_1d)
        w_1d_s   = np.fft.fftshift(w_1d)

        fig_d, axes_d = plt.subplots(1, 3, figsize=(16, 4))

        # Panel 1: cross-spectrum phase (fftshifted, band only)
        band_shift = np.fft.fftshift(band)
        phi_masked = np.where(band_shift & (np.fft.fftshift(w) > 0.00 * w.max()),
                              phi_shift, np.nan)
        im_ph = axes_d[0].imshow(phi_masked, aspect='auto', cmap='RdBu_r',
                                  extent=[kx_disp[0], kx_disp[-1], kz_disp[-1], kz_disp[0]],
                                  vmin=-np.pi, vmax=np.pi, origin='upper')
        plt.colorbar(im_ph, ax=axes_d[0], label='phase [rad]')
        axes_d[0].set_title(f'Cross-spectrum phase  (mask: {n_mask} px)')
        axes_d[0].set_xlabel('kx [rad/m]');  axes_d[0].set_ylabel('kz [rad/m]')

        # Panel 2: fitted plane
        fitted_masked = np.where(band_shift, fitted_shift, np.nan)
        im_fit = axes_d[1].imshow(fitted_masked, aspect='auto', cmap='RdBu_r',
                                   extent=[kx_disp[0], kx_disp[-1], kz_disp[-1], kz_disp[0]],
                                   vmin=-np.pi, vmax=np.pi, origin='upper')
        plt.colorbar(im_fit, ax=axes_d[1], label='phase [rad]')
        axes_d[1].set_title(f'Fitted plane  Δx={dx_est:+.4f} m  φ₀={phi_0:+.3f} rad')
        axes_d[1].set_xlabel('kx [rad/m]');  axes_d[1].set_ylabel('kz [rad/m]')

        # Panel 3: 1-D kx slice — measured vs fitted
        in_band = np.abs(kx_1d_s) < 1.5 * kz_c
        axes_d[2].scatter(kx_1d_s[in_band & (w_1d_s > 0)],
                          phi_1d_s[in_band & (w_1d_s > 0)],
                          c=w_1d_s[in_band & (w_1d_s > 0)],
                          cmap='viridis', s=20, label='measured (weighted)')
        kx_fit = kx_1d_s[in_band]
        axes_d[2].plot(kx_fit, kx_fit * dx_est + phi_0, 'r-', lw=1.5, label='fitted slope')
        axes_d[2].axhline(0, color='k', lw=0.5, ls='--')
        axes_d[2].set_xlabel('kx [rad/m]');  axes_d[2].set_ylabel('phase [rad]')
        axes_d[2].set_title('1-D kx phase profile (collapsed over kz)')
        axes_d[2].legend(fontsize=8)

        plt.suptitle(f'Diagnostic: prof_{ref_run} → prof_{run_b}', y=1.02)
        plt.tight_layout()
        fig_d.savefig(roi_out / f'{ROI_METHOD}_diag_{ref_run}_to_{run_b}.png',
                      dpi=150, bbox_inches='tight')
        plt.show();  plt.close(fig_d)

    results[(ref_run, run_b)] = {
        'dx': dx_est, 'dz': dz_est, 'phi_0': phi_0,
        'roi_m':  (z_roi_bot, z_roi_top, x_roi_lo, x_roi_hi),
        'roi_px': (z0, z1, x0, x1),
        'n_mask': n_mask, 'source': roi_source,
    }
    print(f'prof_{ref_run}→{run_b}  [{roi_source}]:  '
          f'Δx={dx_est:+.4f} m  φ₀={phi_0:+.4f} rad  '
          f'mask={n_mask} px  crop={Nz}×{Nx} px  '
          f'ROI depth=[{z_roi_bot:.1f},{z_roi_top:.1f}] m  '
          f'radial=[{x_roi_lo:.2f},{x_roi_hi:.2f}] m')

# ── Summary plot ─────────────────────────────────────────────────────────────────────
if results:
    pair_labels = [f'{a}→{b}' for a, b in results]
    dx_vals  = [results[k]['dx']    for k in results]
    phi_vals = [results[k]['phi_0'] for k in results]

    fig2, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    ax1.plot(dx_vals, 'o-', color='steelblue')
    ax1.axhline(0, color='k', lw=0.6, ls='--')
    ax1.set_ylabel('Δx (radial shift) [m]')
    mode_str = f'vs prof_{COMPARE_TO}' if COMPARE_TO is not None else 'consecutive pairs'
    ax1.set_title(f'WLS fluid-front shift — {ROI_METHOD.capitalize()} ({mode_str})')
    ax2.plot(np.degrees(phi_vals), 's-', color='darkorange')
    ax2.axhline(0, color='k', lw=0.6, ls='--')
    ax2.set_ylabel('φ₀ [°]')
    ax2.set_xlabel('Pair')
    ax2.set_xticks(range(len(pair_labels)))
    ax2.set_xticklabels(pair_labels, rotation=30, ha='right')
    plt.tight_layout()
    fig2.savefig(roi_out / f'{ROI_METHOD}_dx_summary.png', dpi=150)
    plt.show();  plt.close(fig2)

print(f'\nDone — {len(results)} pair(s).  Output: {roi_out}')

In [ ]:
# ── Manual pair selection + ROI definition — Back-propagation Ez focus frames ────────
#
# MANUAL_PAIRS_BP  — list of (slug_a, slug_b) string tuples from the `completed` dict.
#                    e.g. ('prof_1', 'prof_4').
#
# MANUAL_ROIS_BP   — per-pair ROI in gprMax physical coordinates (metres):
#                      (slug_a, slug_b): (depth_min_m, depth_max_m, radial_min_m, radial_max_m)
#                    depth_min/max refer to the gprMax x-axis (borehole source positions,
#                    0 → 86 m; sources sit between ~62 m and 85 m in this dataset).
#                    radial_min/max refer to the gprMax y-axis (0 → ~14 m from borehole).
#                    Set a pair's value to None to auto-detect from the monogenic envelope.
#
# FOCUS_IDX_OFFSET_BP — snapshot index offset, must match the value in the snapshot cell.
# BP_FORCE_DZ_ZERO    — True: fit only Δx + φ₀ (fluid front is lateral only).
# BP_ROI_THRESH       — envelope threshold for auto-detected pairs (0–1).
# ─────────────────────────────────────────────────────────────────────────────────────

MANUAL_PAIRS_BP = [
    ('prof_1',  'prof_2'),
    ('prof_2',  'prof_3'),
    ('prof_3',  'prof_4'),
    ('prof_4', 'prof_5'),
    ('prof_5', 'prof_7'),
    ('prof_7', 'prof_8'),
    ('prof_8', 'prof_9'),
    ('prof_9', 'prof_10')
]

MANUAL_ROIS_BP = {
    ('prof_1',  'prof_2'): (70, 78, 5.0, 6.5),
    ('prof_2',  'prof_3'): (70, 78, 5.0, 6.5),
    ('prof_3',  'prof_4'): (70, 78, 5.0, 6.5),
    ('prof_4', 'prof_5'): (70, 78, 5.0, 6.5),
    ('prof_5', 'prof_7'): (70, 78, 5.0, 6.5),
    ('prof_7', 'prof_8'): (70, 78, 5.0, 6.5),
    ('prof_8', 'prof_9'): (70, 78, 5.0, 6.5),
    ('prof_9', 'prof_10'): (70, 78, 5.0, 6.5)
}

FOCUS_IDX_OFFSET_BP = 26    # must match offset used in the snapshot cell above
BP_FORCE_DZ_ZERO    = True
BP_ROI_THRESH       = 0.5

# ─────────────────────────────────────────────────────────────────────────────────────

bp_roi_out = OUT_DIR / 'roi_phase' / 'backprop'
bp_roi_out.mkdir(parents=True, exist_ok=True)

def _load_bp_ez(slug):
    """Load the focus-frame Ez array for a given backprop slug.
    Returns (ez, t_actual_ns, depth_axis_m, radial_axis_m, dx_m).
    """
    snap_files = completed.get(slug)
    if not snap_files:
        return None
    import pyvista
    snap_times_ns = t_start_ns + np.arange(len(snap_files)) * snap_step * dt_ns_bp
    idx = min(int(np.argmin(np.abs(snap_times_ns - t_focus_ns))) + FOCUS_IDX_OFFSET_BP,
              len(snap_files) - 1)
    mesh   = pyvista.read(str(snap_files[idx]))
    nx_c   = mesh.dimensions[0] - 1
    ny_c   = mesh.dimensions[1] - 1
    dx_m   = float(mesh.spacing[0])
    e_data = np.array(mesh['E-field'])
    ez     = e_data[:, 2].reshape(ny_c, nx_c).T      # (nx_c, ny_c): rows=depth, cols=radial
    depth_axis  = np.linspace(0, nx_c * dx_m, nx_c)  # gprMax x: 0 → domain_x [m]
    radial_axis = np.linspace(0, ny_c * dx_m, ny_c)  # gprMax y: 0 → domain_y [m]
    return ez, snap_times_ns[idx], depth_axis, radial_axis, dx_m

def _phys_to_pix_bp(depth_axis, radial_axis, d_min, d_max, r_min, r_max):
    """Convert physical-coordinate ROI to pixel indices for the backprop Ez array.
    depth_axis is INCREASING (row 0 = 0 m, last row = domain_x ≈ 86 m).
    Returns (d0, d1, r0, r1) as exclusive-end integer indices.
    """
    row_mask = (depth_axis  >= d_min) & (depth_axis  <= d_max)
    col_mask = (radial_axis >= r_min) & (radial_axis <= r_max)
    rows = np.where(row_mask)[0]
    cols = np.where(col_mask)[0]
    if rows.size == 0 or cols.size == 0:
        raise ValueError(
            f'ROI ({d_min}–{d_max} m depth, {r_min}–{r_max} m radial) '
            f'is outside grid  depth=[0, {depth_axis[-1]:.1f}] m  '
            f'radial=[0, {radial_axis[-1]:.1f}] m'
        )
    return int(rows[0]), int(rows[-1]) + 1, int(cols[0]), int(cols[-1]) + 1

bp_results = {}
for slug_a, slug_b in MANUAL_PAIRS_BP:
    res_a = _load_bp_ez(slug_a)
    res_b = _load_bp_ez(slug_b)
    if res_a is None or res_b is None:
        missing = slug_a if res_a is None else slug_b
        print(f'[{slug_a}→{slug_b}] {missing} not in completed dict — skipping.')
        continue

    ez_a, t_a, depth_ax, radial_ax, dx_m = res_a
    ez_b, t_b, _,        _,         _    = res_b

    # Align shapes (should be identical, but guard against edge cases)
    nd = min(ez_a.shape[0], ez_b.shape[0])
    nr = min(ez_a.shape[1], ez_b.shape[1])
    ez_a, ez_b   = ez_a[:nd, :nr], ez_b[:nd, :nr]
    depth_ax     = depth_ax[:nd]
    radial_ax    = radial_ax[:nr]
    diff = ez_b - ez_a
    env  = _monogenic_envelope(diff)

    # Resolve ROI
    roi_spec = MANUAL_ROIS_BP.get((slug_a, slug_b), None)
    if roi_spec is not None:
        d_min, d_max, r_min, r_max = roi_spec
        d0, d1, r0, r1 = _phys_to_pix_bp(depth_ax, radial_ax, d_min, d_max, r_min, r_max)
        roi_source = 'manual'
    else:
        d0, d1, r0, r1 = _roi_from_envelope(env, BP_ROI_THRESH)
        roi_source = f'auto (thresh={BP_ROI_THRESH})'

    d_roi_lo = float(depth_ax[d0])
    d_roi_hi = float(depth_ax[d1 - 1])
    r_roi_lo = float(radial_ax[r0])
    r_roi_hi = float(radial_ax[r1 - 1])

    # extent convention: [x_min, x_max, y_min, y_max] with origin='lower' + invert_yaxis
    # → x = radial, y = depth with 0 at top after invert
    extent = [0, float(radial_ax[-1]), 0, float(depth_ax[-1])]
    vmax_d = np.percentile(np.abs(diff), 98)
    vmax_e = np.percentile(env, 98)

    fig, (ax_d, ax_e) = plt.subplots(1, 2, figsize=(14, 5))

    im_d = ax_d.imshow(diff, aspect='auto', cmap='RdBu_r',
                        extent=extent, origin='lower',
                        vmin=-vmax_d, vmax=vmax_d)
    plt.colorbar(im_d, ax=ax_d, label='ΔEz [V/m]')
    ax_d.invert_yaxis()
    ax_d.add_patch(Rectangle((r_roi_lo, d_roi_lo),
                              width=r_roi_hi - r_roi_lo,
                              height=d_roi_hi - d_roi_lo,
                              lw=1.5, edgecolor='yellow', facecolor='none'))
    ax_d.xaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_d.yaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_d.grid(True, color='white', linewidth=0.3, alpha=0.4)
    ax_d.set_title(f'Back-prop ΔEz: {slug_b} − {slug_a}  [{roi_source}]')
    ax_d.set_xlabel('Radial distance (m)')
    ax_d.set_ylabel('Depth (m)')

    im_e = ax_e.imshow(env, aspect='auto', cmap='inferno',
                        extent=extent, origin='lower',
                        vmin=0, vmax=vmax_e)
    plt.colorbar(im_e, ax=ax_e, label='Monogenic envelope [a.u.]')
    ax_e.invert_yaxis()
    ax_e.add_patch(Rectangle((r_roi_lo, d_roi_lo),
                              width=r_roi_hi - r_roi_lo,
                              height=d_roi_hi - d_roi_lo,
                              lw=1.5, edgecolor='cyan', facecolor='none'))
    ax_e.xaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_e.yaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_e.grid(True, color='white', linewidth=0.3, alpha=0.4)
    ax_e.set_title(f'Monogenic envelope  [{roi_source}]')
    ax_e.set_xlabel('Radial distance (m)')
    ax_e.set_ylabel('Depth (m)')

    plt.tight_layout()
    fname = f'bp_roi_{slug_a}_to_{slug_b}.png'
    fig.savefig(bp_roi_out / fname, dpi=150)
    plt.show()
    plt.close(fig)

    # WLS phase-plane fit — uniform grid so dz_g = dx_g = dx_m
    # kz_c uses half-velocity (eps_r * 4 → v_half = v/2) since gprMax runs at v/2
    kz_c_bp = 2.0 * np.pi * f0_mig / (v / 2)
    base_crop = ez_a[d0:d1, r0:r1]
    mon_crop  = ez_b[d0:d1, r0:r1]
    dz_est, dx_est, phi_0 = _estimate_shift_2d(
        base_crop, mon_crop, dx_m, dx_m, kz_c_bp, force_dz_zero=BP_FORCE_DZ_ZERO)

    bp_results[(slug_a, slug_b)] = {
        'dx': dx_est, 'dz': dz_est, 'phi_0': phi_0,
        'roi_m':  (d_roi_lo, d_roi_hi, r_roi_lo, r_roi_hi),
        'roi_px': (d0, d1, r0, r1),
        'source': roi_source,
    }
    print(f'{slug_a}→{slug_b}  [{roi_source}]:  '
          f'Δx={dx_est:+.4f} m  φ₀={phi_0:+.4f} rad  '
          f'ROI depth=[{d_roi_lo:.1f}, {d_roi_hi:.1f}] m  '
          f'radial=[{r_roi_lo:.2f}, {r_roi_hi:.2f}] m  '
          f'size={d1-d0}×{r1-r0} px')

# ── Summary plot ─────────────────────────────────────────────────────────────────────
if bp_results:
    pair_labels = [f'{a.split("_")[1]}→{b.split("_")[1]}' for a, b in bp_results]
    dx_vals  = [bp_results[k]['dx']    for k in bp_results]
    phi_vals = [bp_results[k]['phi_0'] for k in bp_results]

    fig2, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    ax1.plot(dx_vals, 'o-', color='steelblue')
    ax1.axhline(0, color='k', lw=0.6, ls='--')
    ax1.set_ylabel('Δx (radial shift) [m]')
    ax1.set_title('Localised WLS fluid-front shift — Back-propagation Ez')
    ax2.plot(np.degrees(phi_vals), 's-', color='darkorange')
    ax2.axhline(0, color='k', lw=0.6, ls='--')
    ax2.set_ylabel('φ₀ [°]')
    ax2.set_xlabel('Pair')
    ax2.set_xticks(range(len(pair_labels)))
    ax2.set_xticklabels(pair_labels, rotation=30, ha='right')
    plt.tight_layout()
    fig2.savefig(bp_roi_out / 'bp_dx_summary.png', dpi=150)
    plt.show()
    plt.close(fig2)

print(f'\nDone — {len(bp_results)} pair(s) analysed.  Output: {bp_roi_out}')